In [36]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import joblib


In [37]:
df = pd.read_csv("../data/logs.csv", usecols=["response_time_ms", "status_code", "level", "service"])
df.head()

y = pd.read_csv("../data/logs.csv", usecols=["is_anomaly"])

df["timestamp"] = pd.read_csv("../data/logs.csv", usecols=["timestamp"])["timestamp"]
df["hour_of_day"] = pd.to_datetime(df["timestamp"]).dt.hour
df.drop(columns=["timestamp"], inplace=True)

In [38]:
# Creating Encoders
level_encoder = LabelEncoder()
service_encoder = LabelEncoder()

# Encode columns
df["level_encoded"] = level_encoder.fit_transform(df["level"])
df["service_encoded"] = service_encoder.fit_transform(df["service"])

# Drop original categorical columns
df.drop(columns=["level", "service"], inplace=True)

# Confirm final dataframe
print(df.head())

   response_time_ms  status_code  hour_of_day  level_encoded  service_encoded
0               499          200           17              2                3
1              7121          500           17              2                3
2               640          200           17              2                2
3               328          200           17              2                1
4               505          200           17              2                3


In [39]:
# Scale the features
scaler = StandardScaler()
X = scaler.fit_transform(df)

# Create Isolation Forest model
model = IsolationForest(
    contamination=0.12,
    random_state=42,
    max_samples=512,
    n_estimators=200
)

# Train model
model.fit(X)

print("Model trained")

Model trained


In [40]:
# Generate predictions
predictions = model.predict(X)

# Convert Isolation Forest output:
# -1 = anomaly -> True
#  1 = normal -> False
predictions = (predictions == -1)

# Convert y dataframe to series
#y = y["is_anomaly"]

# Evaluation
print("Classification Report:")
print(classification_report(y, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y, predictions))

Classification Report:
              precision    recall  f1-score   support

       False       0.98      0.96      0.97      9023
        True       0.68      0.83      0.75       977

    accuracy                           0.94     10000
   macro avg       0.83      0.89      0.86     10000
weighted avg       0.95      0.94      0.95     10000


Confusion Matrix:
[[8636  387]
 [ 164  813]]


In [41]:
joblib.dump(model, "../backend/model.pkl")
joblib.dump(scaler, "../backend/scaler.pkl")
joblib.dump(level_encoder, "../backend/level_encoder.pkl")
joblib.dump(service_encoder, "../backend/service_encoder.pkl")

print("Model, scaler and encoders saved.")

Model, scaler and encoders saved.
